# Price Elasticity of Demand Analysis

**Analyzing how beef prices affect quantity demanded using economic elasticity concepts.**

## Project Overview

This project explores the Price Elasticity of Demand (PED) using historical beef price and quantity data. PED measures how sensitive consumer demand is to price changes — a fundamental concept in economics and business pricing strategy.

## Learning Objectives

1. Understand and calculate Price Elasticity of Demand
2. Differentiate between elastic, inelastic, and unit elastic demand
3. Visualize demand curves and elasticity
4. Apply log-log regression for constant elasticity estimation
5. Interpret economic significance of elasticity values

## Business / Research Problem

Pricing decisions are critical for businesses and policymakers. Understanding elasticity helps:
- **Businesses** set profit-maximizing prices
- **Governments** predict tax revenue impacts
- **Economists** model market behavior

**Key question:** *How elastic is the demand for beef, and what does this tell us about consumer behavior?*

## Why This Analysis Matters

Elasticity is central to microeconomics. Knowing whether demand is elastic or inelastic determines whether raising prices increases or decreases total revenue.

## Dataset Overview

| Feature | Type | Description |
|---------|------|-------------|
| `Year` | Numeric | Year of observation |
| `Quarter` | Numeric | Quarter (1-4) |
| `Quantity` | Numeric | Quantity demanded |
| `Price` | Numeric | Price per unit |

**Note:** The dataset uses the local `beef.csv` file in this project directory.

## Dataset Source & License Notes

- **Source:** US Department of Agriculture / Economics textbook data
- **License:** Public domain (government data)
- **File:** `beef.csv` in the project directory

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
DATA_FILE = 'beef.csv'
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

This dataset is included locally as `beef.csv`.

In [ ]:
import os

df = pd.read_csv(DATA_FILE)
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head(10)

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
df.describe()

## Data Cleaning

In [ ]:
df = df.drop_duplicates()

# Create time period column
df['Period'] = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)

# Calculate percentage changes
df['Price_pct_change'] = df['Price'].pct_change() * 100
df['Qty_pct_change'] = df['Quantity'].pct_change() * 100

# Log transforms for elasticity
df['log_price'] = np.log(df['Price'])
df['log_qty'] = np.log(df['Quantity'])

print(f'Prepared {len(df)} observations')
df.head()

## Exploratory Data Analysis

In [ ]:
print('=== Price Summary ===')
print(f'Mean: {df["Price"].mean():.2f}')
print(f'Std:  {df["Price"].std():.2f}')
print(f'Range: {df["Price"].min():.2f} - {df["Price"].max():.2f}')

print(f'\n=== Quantity Summary ===')
print(f'Mean: {df["Quantity"].mean():.2f}')
print(f'Std:  {df["Quantity"].std():.2f}')
print(f'Range: {df["Quantity"].min():.2f} - {df["Quantity"].max():.2f}')

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Price'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price')

axes[1].hist(df['Quantity'], bins=20, color='coral', edgecolor='white')
axes[1].set_title('Quantity Distribution')
axes[1].set_xlabel('Quantity')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

The demand curve shows the relationship between price and quantity.

In [ ]:
# Demand curve: Price vs Quantity
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(df['Quantity'], df['Price'], alpha=0.6, color='steelblue', s=30)
# Add trend line
z = np.polyfit(df['Quantity'], df['Price'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Quantity'].min(), df['Quantity'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', alpha=0.8, label=f'Trend line')
axes[0].set_xlabel('Quantity Demanded')
axes[0].set_ylabel('Price')
axes[0].set_title('Demand Curve (Price vs Quantity)')
axes[0].legend()

# Log-log demand curve
axes[1].scatter(df['log_qty'], df['log_price'], alpha=0.6, color='coral', s=30)
z_log = np.polyfit(df['log_qty'], df['log_price'], 1)
p_log = np.poly1d(z_log)
x_log = np.linspace(df['log_qty'].min(), df['log_qty'].max(), 100)
axes[1].plot(x_log, p_log(x_log), 'r--', alpha=0.8, label=f'Slope (elasticity) = {z_log[0]:.3f}')
axes[1].set_xlabel('Log(Quantity)')
axes[1].set_ylabel('Log(Price)')
axes[1].set_title('Log-Log Demand Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## Feature-Specific Insights

### Calculating Price Elasticity of Demand

PED = (% Change in Quantity) / (% Change in Price)

- |PED| > 1: Elastic (quantity is sensitive to price)
- |PED| < 1: Inelastic (quantity is insensitive to price)
- |PED| = 1: Unit elastic

In [ ]:
# Point elasticity for each observation
valid = df.dropna(subset=['Price_pct_change', 'Qty_pct_change'])
valid = valid[valid['Price_pct_change'] != 0]  # avoid division by zero
valid['PED'] = valid['Qty_pct_change'] / valid['Price_pct_change']

print('=== Point Elasticity Summary ===')
print(f'Mean PED: {valid["PED"].mean():.4f}')
print(f'Median PED: {valid["PED"].median():.4f}')
print(f'\nInterpretation: {"Inelastic" if abs(valid["PED"].median()) < 1 else "Elastic"} demand')

# Log-log regression for constant elasticity
slope, intercept, r_value, p_value, std_err = stats.linregress(df['log_price'], df['log_qty'])
print(f'\n=== Log-Log Regression Elasticity ===')
print(f'Elasticity (slope): {slope:.4f}')
print(f'R-squared: {r_value**2:.4f}')
print(f'p-value: {p_value:.2e}')
print(f'Interpretation: A 1% increase in price leads to a {abs(slope):.2f}% {"decrease" if slope < 0 else "increase"} in quantity')

## Statistical Checks / Hypothesis Testing

In [ ]:
# Test: Is the correlation between price and quantity significant?
r, p = stats.pearsonr(df['Price'], df['Quantity'])
print(f'Pearson correlation: r={r:.4f}, p={p:.2e}')
print(f'Result: {"Significant" if p < 0.05 else "Not significant"} negative correlation')

# Test: Is elasticity significantly different from -1 (unit elastic)?
t_stat = (slope - (-1)) / std_err
p_val_unit = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(df) - 2))
print(f'\nTest: Elasticity != -1 (unit elastic)')
print(f't-stat: {t_stat:.4f}, p-value: {p_val_unit:.4f}')
print(f'Result: Elasticity is {"significantly different from" if p_val_unit < 0.05 else "not significantly different from"} unit elastic')

## Visual Analysis

In [ ]:
# Time series of price and quantity
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(range(len(df)), df['Price'], 'b-', alpha=0.7, label='Price')
ax1.set_xlabel('Time Period')
ax1.set_ylabel('Price', color='b')

ax2 = ax1.twinx()
ax2.plot(range(len(df)), df['Quantity'], 'r-', alpha=0.7, label='Quantity')
ax2.set_ylabel('Quantity', color='r')

fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.95))
plt.title('Price and Quantity Over Time')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue analysis
df['Revenue'] = df['Price'] * df['Quantity']

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(range(len(df)), df['Revenue'], 'g-', alpha=0.7, linewidth=2)
ax.fill_between(range(len(df)), df['Revenue'], alpha=0.2, color='green')
ax.set_title('Total Revenue Over Time (Price x Quantity)')
ax.set_xlabel('Time Period')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()

## Key Findings

1. **Price and quantity are negatively correlated** — consistent with the Law of Demand
2. **Demand elasticity estimate** tells us whether the market is price-sensitive
3. **The log-log model provides constant elasticity** — a key assumption in economic modeling
4. **Price changes over time** reflect supply/demand shifts in the beef market
5. **Revenue implications depend on elasticity** — raising prices on inelastic goods increases revenue

## Limitations

1. **Omitted variable bias:** Income, substitute prices, and preferences are not controlled for
2. **Supply vs demand shifts:** We observe equilibrium points, not the demand curve itself
3. **Time-varying elasticity:** Elasticity may change over the sample period
4. **Aggregated data:** Individual consumer behavior is masked
5. **No seasonal adjustment:** Quarterly patterns are not controlled for

## Recommendations / Next Steps

1. Include income and substitute good prices for a proper demand model
2. Use instrumental variables to address endogeneity
3. Estimate time-varying elasticity with rolling regressions

### How to Extend This Analysis
- Compare elasticity across different food products
- Build a simultaneous equations model for supply and demand
- Apply this framework to your own product pricing data

## Common Mistakes

1. **Confusing movements along vs shifts of the demand curve**
2. **Ignoring the sign:** PED is negative by convention; absolute value determines elastic/inelastic
3. **Using level data for constant elasticity:** Log-log specification is needed
4. **Assuming elasticity is constant everywhere:** Point elasticity varies along a linear demand curve
5. **Claiming causation from correlation:** Price-quantity correlation doesn't prove the demand relationship

## Mini Challenge / Exercises

1. Calculate arc elasticity between the first and last data points
2. What price would maximize total revenue based on the estimated demand curve?
3. Create a quarterly seasonality analysis — does demand vary by quarter?
4. Simulate what happens to revenue if price increases by 10% (using estimated elasticity)
5. Plot the relationship between price level and point elasticity

In [ ]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Price Elasticity of Demand** measures consumer responsiveness to price changes
- **Log-log regression** is the standard method for estimating constant elasticity
- **Beef demand** shows a typical downward-sloping demand curve
- **Elasticity determines revenue strategy:** raise/lower prices depending on whether demand is elastic or inelastic
- **This framework applies to any product** — adapt the methodology to your industry

Understanding elasticity is fundamental to pricing strategy, tax policy, and market analysis.